## 02 - Métricas Técnicas e de Negócio

**Tech Challenge — Fase 1 | Machine Learning Engineering — FIAP**

**Projeto:** Customer Churn Prediction

**Grupo:** 56

---

### Objetivo

Este notebook define o protocolo de avaliação técnica e de negócio utilizado para comparar os modelos de previsão de churn.

São estabelecidas:

- a classe positiva e a estratégia de divisão dos dados;
- as métricas técnicas utilizadas na avaliação;
- a métrica principal para comparação dos modelos;
- a interpretação dos Falsos Positivos e Falsos Negativos;
- os indicadores de impacto potencial no negócio;
- os critérios para seleção do modelo campeão.

As definições estabelecidas neste notebook serão aplicadas de forma consistente aos experimentos de modelagem realizados nas etapas seguintes.

---

In [1]:
# ==================================================================
# ETAPA 1 - CONFIGURAÇÃO DO AMBIENTE E IMPORTAÇÃO DAS BIBLIOTECAS
# ==================================================================

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [2]:
# ============================================================
# ETAPA 2 - CARREGAMENTO DA BASE DE DADOS
# ============================================================
# Localizando a raiz do projeto a partir do diretório atual e
# carregando a base armazenada na camada data/raw.
#
# A identificação dinâmica da raiz evita dependência de caminhos
# absolutos específicos da máquina local e melhora a portabilidade
# do notebook.
# ============================================================

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset não encontrado em: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print("Base carregada com sucesso.")
print(f"Quantidade de registros: {df.shape[0]:,}".replace(",", "."))
print(f"Quantidade de variáveis: {df.shape[1]}")
print(f"Arquivo utilizado: {DATA_PATH.name}")
print(f"Raiz do projeto: {PROJECT_ROOT}")

display(df.head())

Base carregada com sucesso.
Quantidade de registros: 7.043
Quantidade de variáveis: 21
Arquivo utilizado: WA_Fn-UseC_-Telco-Customer-Churn.csv
Raiz do projeto: /Users/biancafirmino/churn-prediction-mle


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# ============================================================
# ETAPA 3 - PREPARAÇÃO DA VARIÁVEL ALVO
# ============================================================
# Validando e convertendo a variável alvo Churn para formato
# numérico, necessário para o treinamento e avaliação dos
# modelos de classificação.
#
# Mapeamento:
#   No  -> 0 (Permaneceu)
#   Yes -> 1 (Cancelou)
# ============================================================

print("Valores encontrados na variável alvo:")
display(df["Churn"].value_counts(dropna=False))

# Conversão da variável alvo
y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# Validação da conversão
assert y.isna().sum() == 0, (
    "Foram encontrados valores inesperados na variável Churn."
)

print("\nDistribuição da variável alvo após conversão:")
display(
    y.value_counts()
     .rename(index={0: "Permaneceu (0)", 1: "Cancelou (1)"})
)

Valores encontrados na variável alvo:


Churn
No     5174
Yes    1869
Name: count, dtype: int64


Distribuição da variável alvo após conversão:


Churn
Permaneceu (0)    5174
Cancelou (1)      1869
Name: count, dtype: int64

In [4]:
# ============================================================
# ETAPA 3.1 - DEFINIÇÃO DAS VARIÁVEIS EXPLICATIVAS
# ============================================================
# Separando as variáveis utilizadas como entrada do modelo.
#
# customerID é removida por representar apenas um identificador
# único do cliente, sem significado preditivo direto.
#
# Churn é removida por ser a variável alvo que o modelo deverá
# aprender a prever.
# ============================================================

X = df.drop(columns=["customerID", "Churn"])

print(f"Quantidade de registros: {X.shape[0]:,}".replace(",", "."))
print(f"Quantidade de variáveis explicativas: {X.shape[1]}")

display(X.head())

Quantidade de registros: 7.043
Quantidade de variáveis explicativas: 19


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


#### Conclusão da Definição das Variáveis

A variável **Churn** foi definida como alvo binário, utilizando:

- 0 → Cliente permaneceu;
- 1 → Cliente cancelou.

A variável **customerID** foi removida das variáveis explicativas por representar apenas um identificador único, sem significado preditivo.

A matriz final será utilizada em todas as etapas de modelagem.

---

In [5]:
# ============================================================
# ETAPA 4 - DEFINIÇÃO DO PROTOCOLO DE DIVISÃO DOS DADOS
# ============================================================
# Separando os dados em conjuntos de treino (80%) e teste (20%).
#
# A estratificação pela variável alvo (stratify=y) preserva
# aproximadamente a mesma proporção de clientes que permaneceram
# e cancelaram em ambos os conjuntos.
#
# O random_state garante a reprodutibilidade da divisão,
# permitindo reproduzir os mesmos conjuntos em execuções futuras.
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dimensões dos conjuntos:")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

Dimensões dos conjuntos:
X_train: (5634, 19)
X_test:  (1409, 19)
y_train: (5634,)
y_test:  (1409,)


In [6]:
# ============================================================
# ETAPA 4.1 - Validação da estratificação da variável alvo.
# ============================================================
# Comparando a distribuição percentual da variável alvo na base
# completa, no conjunto de treino e no conjunto de teste para
# verificar se a proporção das classes foi preservada.
# ============================================================

distribuicao_classes = pd.DataFrame({
    "Base Completa (%)": y.value_counts(normalize=True).sort_index() * 100,
    "Treino (%)": y_train.value_counts(normalize=True).sort_index() * 100,
    "Teste (%)": y_test.value_counts(normalize=True).sort_index() * 100
})

distribuicao_classes.index = [
    "Permaneceu (0)",
    "Cancelou (1)"
]

display(distribuicao_classes.round(2))

,Base Completa (%),Treino (%),Teste (%)
Permaneceu (0),73.46,73.46,73.46
Cancelou (1),26.54,26.54,26.54


#### Conclusão do Protocolo de Divisão

Foi definido um protocolo de divisão estratificada utilizando **80% dos registros para treinamento** e **20% para teste**, mantendo a proporção da variável alvo em ambos os conjuntos.

O parâmetro **random_state=42** garante reprodutibilidade dos experimentos.

Esse protocolo será utilizado em todos os modelos desenvolvidos neste projeto.

---


### ETAPA 5 - Definição das Métricas Técnicas

A avaliação de um modelo de classificação de churn exige a utilização de métricas que considerem não apenas a quantidade total de previsões corretas, mas também os diferentes tipos de erro cometidos pelo modelo.

Neste projeto, a classe positiva é definida como:

- **0 — Permaneceu:** cliente que não apresentou churn.
- **1 — Cancelou:** cliente que apresentou churn.

A Análise Exploratória dos Dados identificou um desbalanceamento moderado entre as classes, com aproximadamente **73,46% dos clientes permanecendo** e **26,54% cancelando** o serviço.

Por esse motivo, a acurácia não será utilizada isoladamente como critério de seleção dos modelos.

As métricas técnicas adotadas serão:

- **Accuracy (Acurácia):** desempenho global das classificações;
- **Precision (Precisão):** confiabilidade das previsões de churn;
- **Recall (Sensibilidade):** capacidade de identificar clientes que realmente cancelarão;
- **F1-Score:** equilíbrio entre Precision e Recall;
- **ROC-AUC:** capacidade global de discriminação entre as classes;
- **PR-AUC:** desempenho na identificação da classe positiva, considerando o desbalanceamento existente.

A avaliação será realizada utilizando múltiplas métricas, pois cada uma representa uma perspectiva diferente do desempenho do modelo.

---

In [7]:
# ============================================================
# ETAPA 5.1 - ESTRUTURA DAS MÉTRICAS TÉCNICAS
# ============================================================
# Organizando as métricas que serão utilizadas para avaliar e
# comparar os modelos desenvolvidos ao longo do projeto.
# ============================================================

metricas_tecnicas = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC",
        "PR-AUC"
    ],
    "Objetivo": [
        "Avaliar a proporção total de previsões corretas",
        "Avaliar a confiabilidade das previsões de churn",
        "Identificar corretamente clientes que realmente cancelarão",
        "Equilibrar Precision e Recall",
        "Avaliar a capacidade global de discriminação entre as classes",
        "Avaliar o desempenho sobre a classe positiva em cenário desbalanceado"
    ],
    "Prioridade": [
        "Secundária",
        "Secundária",
        "Alta",
        "Alta",
        "Alta",
        "Principal"
    ]
})

display(metricas_tecnicas)

,Métrica,Objetivo,Prioridade
0,Accuracy,Avaliar a proporção total de previsões corretas,Secundária
1,Precision,Avaliar a confiabilidade das previsões de churn,Secundária
2,Recall,Identificar corretamente clientes que realment...,Alta
3,F1-Score,Equilibrar Precision e Recall,Alta
4,ROC-AUC,Avaliar a capacidade global de discriminação e...,Alta
5,PR-AUC,Avaliar o desempenho sobre a classe positiva e...,Principal


---

### ETAPA 5.2 - Definição da Métrica Técnica Principal

A análise baseada na **Precision-Recall** será utilizada como principal referência técnica para comparação dos modelos desenvolvidos neste projeto.

A implementação será realizada por meio da métrica **Average Precision (AP)**, calculada com `average_precision_score`, que resume o desempenho da curva Precision-Recall em um único valor.

Essa escolha considera o desbalanceamento observado na variável alvo, em que aproximadamente **26,54% dos clientes pertencem à classe positiva (Churn = 1)**.

Nesse contexto, métricas baseadas em Precision e Recall permitem avaliar de forma mais adequada a capacidade do modelo de identificar clientes em risco sem aumentar excessivamente o número de Falsos Positivos.

As demais métricas serão utilizadas de forma complementar:

- **Recall:** capacidade de identificar clientes que realmente apresentaram churn;
- **Precision:** confiabilidade das previsões positivas;
- **F1-Score:** equilíbrio entre Precision e Recall;
- **ROC-AUC:** capacidade geral de separação entre as classes;
- **Accuracy:** utilizada apenas como métrica complementar.

---

In [8]:
# ============================================================
# ETAPA 5.3 - REFERÊNCIA PARA A CURVA PRECISION-RECALL
# ============================================================
# A prevalência da classe positiva representa o desempenho
# esperado de um classificador aleatório na métrica
# Precision-Recall (Average Precision).
#
# Essa referência será utilizada posteriormente para verificar
# se os modelos realmente apresentam ganho preditivo.
# ============================================================

prevalencia_churn = y.mean()

print(
    f"Prevalência de churn na base: "
    f"{prevalencia_churn:.2%}"
)

print(
    f"Referência para Average Precision: "
    f"{prevalencia_churn:.4f}"
)

Prevalência de churn na base: 26.54%
Referência para Average Precision: 0.2654


#### Conclusão das Métricas Técnicas

Foi definida uma estratégia de avaliação composta por métricas complementares, priorizando a análise Precision–Recall devido ao desbalanceamento moderado da variável alvo.

A Average Precision (AP), em conjunto com a curva Precision–Recall, será utilizada como principal referência para comparação dos modelos, enquanto Recall, Precision, F1-Score, ROC-AUC e Accuracy serão utilizadas como métricas de apoio.

Essa abordagem permitirá avaliar os modelos de forma mais consistente tanto sob a perspectiva estatística quanto sob a perspectiva do problema de negócio.

---

### ETAPA 6 - Interpretação dos Erros de Classificação

Além das métricas agregadas, é fundamental compreender os diferentes tipos de acertos e erros que podem ser produzidos pelos modelos de classificação.

Neste projeto, considera-se **Churn = 1** como a classe positiva.

A matriz de confusão será interpretada da seguinte forma:

| Resultado | Interpretação no contexto de churn |
|---|---|
| **Verdadeiro Negativo (TN)** | O modelo prevê que o cliente permanecerá e ele realmente permanece. |
| **Falso Positivo (FP)** | O modelo prevê churn, mas o cliente permaneceria. |
| **Falso Negativo (FN)** | O modelo prevê permanência, mas o cliente realmente cancela. |
| **Verdadeiro Positivo (TP)** | O modelo prevê churn e o cliente realmente cancela. |

Os dois tipos de erro possuem impactos distintos para o negócio.

Um **Falso Positivo (FP)** pode gerar uma ação de retenção desnecessária, como a concessão de descontos, benefícios ou contato comercial para um cliente que não pretendia cancelar.

Um **Falso Negativo (FN)** representa um cliente com risco real de churn que não foi identificado pelo modelo. Nesse cenário, a empresa pode perder a oportunidade de realizar uma ação preventiva de retenção.

Por esse motivo, os Falsos Negativos possuem especial relevância neste projeto. Entretanto, maximizar o Recall indiscriminadamente também pode aumentar o número de Falsos Positivos e, consequentemente, elevar os custos das campanhas de retenção.

A avaliação dos modelos deverá, portanto, buscar um equilíbrio entre a capacidade de identificar clientes em risco e o custo associado às intervenções realizadas.

In [10]:
# ============================================================
# ETAPA 6.1 - TIPOS DE RESULTADOS DA CLASSIFICAÇÃO
# ============================================================
# Estruturando a interpretação dos possíveis resultados de uma
# matriz de confusão no contexto do problema de churn.
# ============================================================

resultados_classificacao = pd.DataFrame({
    "Resultado": [
        "Verdadeiro Negativo (TN)",
        "Falso Positivo (FP)",
        "Falso Negativo (FN)",
        "Verdadeiro Positivo (TP)"
    ],
    "Predição": [
        "Permaneceu (0)",
        "Cancelou (1)",
        "Permaneceu (0)",
        "Cancelou (1)"
    ],
    "Real": [
        "Permaneceu (0)",
        "Permaneceu (0)",
        "Cancelou (1)",
        "Cancelou (1)"
    ],
    "Impacto no Negócio": [
        "Classificação correta de cliente sem churn",
        "Possível custo de retenção desnecessário",
        "Cliente em risco não identificado e possível perda de receita",
        "Cliente em risco corretamente identificado para possível intervenção"
    ]
})

display(resultados_classificacao)

,Resultado,Predição,Real,Impacto no Negócio
0,Verdadeiro Negativo (TN),Permaneceu (0),Permaneceu (0),Classificação correta de cliente sem churn
1,Falso Positivo (FP),Cancelou (1),Permaneceu (0),Possível custo de retenção desnecessário
2,Falso Negativo (FN),Permaneceu (0),Cancelou (1),Cliente em risco não identificado e possível p...
3,Verdadeiro Positivo (TP),Cancelou (1),Cancelou (1),Cliente em risco corretamente identificado par...


### ETAPA 6.2 - Relação entre os Erros e as Métricas

As métricas selecionadas refletem diferentes consequências dos erros de classificação no contexto da previsão de churn:

- **Recall:** mede a proporção de clientes que realmente apresentaram churn e foram corretamente identificados pelo modelo. Um Recall baixo está associado a uma maior proporção de Falsos Negativos, representando clientes em risco que podem deixar de receber uma ação preventiva de retenção.

- **Precision:** mede, entre os clientes classificados como risco de churn, a proporção daqueles que realmente cancelaram. Uma Precision baixa indica maior ocorrência proporcional de Falsos Positivos, podendo resultar em ações de retenção desnecessárias e aumento dos custos operacionais.

- **F1-Score:** representa a média harmônica entre Precision e Recall, permitindo avaliar o equilíbrio entre a identificação dos clientes em risco e a redução de intervenções desnecessárias.

- **ROC-AUC:** avalia a capacidade geral do modelo de discriminar clientes com e sem churn ao longo de diferentes limiares de classificação.

- **PR-AUC:** avalia conjuntamente Precision e Recall em diferentes limiares de decisão, sendo especialmente relevante neste projeto devido ao foco na classe positiva (Churn = 1) e ao desbalanceamento observado entre as classes.

Dessa forma, a seleção do modelo final não será baseada em uma única métrica. A **PR-AUC será utilizada como principal referência técnica**, enquanto Recall, F1-Score e ROC-AUC fornecerão perspectivas complementares sobre o desempenho do modelo.

Essa avaliação multidimensional permitirá considerar tanto a capacidade preditiva dos modelos quanto as consequências dos Falsos Positivos e Falsos Negativos para as futuras estratégias de retenção.

#### Conclusão da Interpretação dos Erros

Os diferentes tipos de erro produzem impactos distintos no contexto da previsão de churn.

Neste projeto, os **Falsos Negativos** receberão atenção especial por representarem clientes que efetivamente cancelam seus contratos sem serem identificados pelo modelo.

Entretanto, Falsos Positivos também serão considerados, pois ações de retenção desnecessárias podem gerar custos adicionais para a empresa.

Por esse motivo, a avaliação dos modelos considerará simultaneamente desempenho técnico e impacto potencial no negócio.

---

### ETAPA 7 - Definição da Métrica de Negócio

Além das métricas técnicas, a avaliação de um modelo de churn deve considerar o impacto econômico das decisões geradas a partir de suas previsões.

Neste projeto, a métrica de negócio será baseada no **valor econômico líquido potencial das ações de retenção**, considerando a identificação de clientes com risco de churn e os custos associados às intervenções.

O conjunto de dados utilizado não fornece diretamente informações como:

- custo de uma campanha de retenção;
- valor financeiro futuro de cada cliente;
- probabilidade real de sucesso de uma ação de retenção;
- custo efetivo decorrente da perda de um cliente.

Por esse motivo, esses valores não serão tratados como fatos observados. Será utilizada uma estrutura parametrizável de simulação, permitindo avaliar diferentes cenários de negócio sem introduzir premissas financeiras como se fossem dados reais.

A lógica econômica considera principalmente:

- **Verdadeiros Positivos (TP):** clientes com churn corretamente identificados, que podem receber uma ação preventiva;
- **Falsos Positivos (FP):** clientes que receberiam uma ação de retenção desnecessariamente;
- **Falsos Negativos (FN):** clientes com churn não identificados, representando oportunidades de retenção perdidas.

Essa abordagem permite conectar o desempenho técnico do modelo ao impacto potencial para o negócio.

### ETAPA 7.1 - Modelo Econômico de Retenção

Para avaliar o impacto econômico potencial do modelo, será utilizada uma estimativa de **Valor Líquido de Retenção**, baseada na seguinte lógica:

**Valor Líquido = Valor do churn potencialmente evitado − Custo das intervenções**

De forma parametrizada:

**Valor Líquido = (TP × taxa de sucesso da retenção × valor do cliente) − ((TP + FP) × custo da intervenção)**

Onde:

- **TP:** clientes com churn corretamente identificados;
- **FP:** clientes sem churn classificados incorretamente como risco;
- **taxa de sucesso da retenção:** proporção estimada de clientes em risco que permaneceriam após uma intervenção;
- **valor do cliente:** valor econômico estimado preservado quando um churn é evitado;
- **custo da intervenção:** custo associado à ação de retenção realizada para cada cliente sinalizado pelo modelo.

Os **Falsos Negativos (FN)** não aparecem diretamente na fórmula de valor realizado, pois representam oportunidades que o modelo deixou de identificar. Entretanto, serão acompanhados por meio do Recall e poderão ser utilizados para estimar o valor potencial perdido.

Como o dataset não fornece os parâmetros financeiros necessários, a métrica será implementada como uma função parametrizável. Os valores utilizados posteriormente deverão ser explicitamente identificados como premissas de simulação e não como informações observadas na base.

In [ ]:
# ============================================================
# ETAPA 7.2 - FUNÇÃO PARAMETRIZÁVEL PARA ESTIMAR O IMPACTO
# ECONÔMICO DAS AÇÕES DE RETENÇÃO
# ============================================================

def calcular_valor_retencao(
    tp,
    fp,
    fn,
    valor_cliente,
    custo_intervencao,
    taxa_sucesso_retencao=1.0,
):
    """
    Calcula indicadores econômicos relacionados às ações de retenção.

    Parameters
    ----------
    tp : int
        Verdadeiros Positivos.
    fp : int
        Falsos Positivos.
    fn : int
        Falsos Negativos.
    valor_cliente : float
        Valor estimado associado à retenção de um cliente.
    custo_intervencao : float
        Custo estimado de cada ação de retenção.
    taxa_sucesso_retencao : float, default=1.0
        Probabilidade de sucesso da ação de retenção (0 a 1).

    Returns
    -------
    dict
        Indicadores econômicos estimados.
    """

    # --------------------------------------------------------
    # Validações dos parâmetros
    # --------------------------------------------------------

    if not 0 <= taxa_sucesso_retencao <= 1:
        raise ValueError(
            "A taxa de sucesso deve estar entre 0 e 1."
        )

    if valor_cliente < 0:
        raise ValueError(
            "O valor do cliente não pode ser negativo."
        )

    if custo_intervencao < 0:
        raise ValueError(
            "O custo da intervenção não pode ser negativo."
        )

    if tp < 0 or fp < 0 or fn < 0:
        raise ValueError(
            "TP, FP e FN não podem ser negativos."
        )

    # --------------------------------------------------------
    # Cálculos
    # --------------------------------------------------------

    churns_evitar = tp * taxa_sucesso_retencao

    valor_churn_evitar = churns_evitar * valor_cliente

    custo_total = (tp + fp) * custo_intervencao

    valor_liquido = valor_churn_evitar - custo_total

    valor_potencial_perdido = fn * valor_cliente

    return {
        "Clientes abordados": tp + fp,
        "Churns potencialmente evitados": churns_evitar,
        "Valor do churn evitado": valor_churn_evitar,
        "Custo total das intervenções": custo_total,
        "Valor líquido estimado": valor_liquido,
        "Valor potencial perdido (FN)": valor_potencial_perdido,
    }

In [12]:
# ============================================================
# ETAPA 7.3 - INDICADORES DE NEGÓCIO
# ============================================================
# Estruturando os indicadores que serão utilizados para conectar
# o desempenho dos modelos ao impacto potencial no negócio.
# ============================================================

metricas_negocio = pd.DataFrame({
    "Indicador": [
        "Clientes abordados",
        "Churns potencialmente evitados",
        "Valor do churn evitado",
        "Custo total das intervenções",
        "Valor líquido estimado",
        "Valor potencial perdido por FN"
    ],
    "Descrição": [
        "Quantidade de clientes sinalizados para ação de retenção",
        "Estimativa de clientes retidos entre os churns corretamente identificados",
        "Valor econômico potencial preservado pela retenção",
        "Custo total das ações realizadas nos clientes sinalizados",
        "Benefício econômico estimado após descontar os custos das intervenções",
        "Valor potencial associado aos clientes com churn não identificados"
    ]
})

display(metricas_negocio)

,Indicador,Descrição
0,Clientes abordados,Quantidade de clientes sinalizados para ação d...
1,Churns potencialmente evitados,Estimativa de clientes retidos entre os churns...
2,Valor do churn evitado,Valor econômico potencial preservado pela rete...
3,Custo total das intervenções,Custo total das ações realizadas nos clientes ...
4,Valor líquido estimado,Benefício econômico estimado após descontar os...
5,Valor potencial perdido por FN,Valor potencial associado aos clientes com chu...


#### Conclusão da Métrica de Negócio

A avaliação de negócio foi estruturada de forma parametrizável, permitindo estimar o impacto econômico potencial das previsões sem assumir valores financeiros inexistentes no dataset.

A principal métrica econômica será o **Valor Líquido Estimado da Retenção**, calculado a partir do valor potencialmente preservado pelos churns evitados menos o custo das intervenções realizadas.

Além disso, serão acompanhados o número de clientes abordados, os churns potencialmente evitados e o valor potencial associado aos Falsos Negativos.

Essa abordagem permite que diferentes modelos e limiares de decisão sejam comparados não apenas por seu desempenho estatístico, mas também pelo impacto potencial de suas decisões sobre o negócio.

Os valores financeiros utilizados em simulações futuras serão tratados explicitamente como premissas de cenário e poderão ser substituídos por dados reais caso essas informações estejam disponíveis em um ambiente de produção.

---

### ETAPA 8 - Critérios para Comparação e Seleção dos Modelos

Após a definição das métricas técnicas e dos indicadores de negócio, torna-se necessário estabelecer um protocolo padronizado para comparação dos modelos que serão desenvolvidos nas próximas etapas.

Para garantir uma avaliação consistente e reproduzível, todos os modelos serão treinados utilizando o mesmo conjunto de treinamento e avaliados exatamente sobre o mesmo conjunto de teste.

Neste projeto serão utilizadas duas referências iniciais:

- **DummyClassifier**: representa a referência mínima de desempenho (baseline ingênuo), indicando o comportamento esperado de um modelo sem capacidade preditiva.
- **Regressão Logística**: será utilizada como baseline estatístico, servindo como primeiro modelo supervisionado para comparação com os modelos mais avançados.

Os modelos candidatos (Random Forest, MLPClassifier e demais abordagens) deverão apresentar desempenho superior ao baseline estatístico para serem considerados alternativas competitivas.

A comparação entre os modelos seguirá os seguintes critérios:

1. **Average Precision (AP) / Curva Precision–Recall** como principal referência técnica;

2. **Recall** como indicador crítico complementar, reduzindo a ocorrência de Falsos Negativos;

3. **F1-Score** para avaliar o equilíbrio entre Precision e Recall;

4. **ROC-AUC** como medida complementar da capacidade discriminatória;

5. **Accuracy** apenas como métrica auxiliar;

6. **Análise da Matriz de Confusão**, considerando separadamente Falsos Positivos e Falsos Negativos;

7. **Avaliação econômica**, utilizando o Valor Líquido Estimado da Retenção.

Essa estratégia garante que a seleção do modelo campeão considere simultaneamente desempenho estatístico, comportamento dos erros e impacto potencial para o negócio.

In [13]:
# ============================================================
# ETAPA 8.1 - PROTOCOLO DE AVALIAÇÃO DOS MODELOS
# ============================================================
# Formalizando os critérios que serão aplicados de maneira
# consistente a todos os modelos desenvolvidos neste projeto.
# ============================================================

protocolo_avaliacao = pd.DataFrame({
    "Critério": [
        "Conjunto de avaliação",
        "Métrica técnica principal",
        "Métricas técnicas complementares",
        "Referência mínima",
        "Baseline estatístico",
        "Classe positiva",
        "Análise dos erros",
        "Avaliação de negócio",
        "Reprodutibilidade"
    ],
    "Definição": [
        "Mesmo conjunto de teste estratificado para todos os modelos",
        "Average Precision (Precision-Recall)",
        "Recall, Precision, F1-Score, ROC-AUC e Accuracy",
        "DummyClassifier",
        "Regressão Logística (Scikit-Learn)",
        "Churn = 1 (Cancelou)",
        "Matriz de Confusão com análise de FP e FN",
        "Valor Líquido Estimado da Retenção",
        "random_state = 42"
    ]
})

display(protocolo_avaliacao)

,Critério,Definição
0,Conjunto de avaliação,Mesmo conjunto de teste estratificado para tod...
1,Métrica técnica principal,Average Precision (Precision-Recall)
2,Métricas técnicas complementares,"Recall, Precision, F1-Score, ROC-AUC e Accuracy"
3,Referência mínima,DummyClassifier
4,Baseline estatístico,Regressão Logística (Scikit-Learn)
5,Classe positiva,Churn = 1 (Cancelou)
6,Análise dos erros,Matriz de Confusão com análise de FP e FN
7,Avaliação de negócio,Valor Líquido Estimado da Retenção
8,Reprodutibilidade,random_state = 42


In [14]:
# ============================================================
# ETAPA 8.2 - FUNÇÃO PADRONIZADA DE AVALIAÇÃO TÉCNICA
# ============================================================
# Criando uma função única para calcular as métricas técnicas
# dos modelos, garantindo consistência entre os experimentos.
#
# A função receberá:
# - valores reais;
# - classes previstas;
# - probabilidades da classe positiva.
# ============================================================

def avaliar_modelo(y_true, y_pred, y_proba):
    """
    Calcula as principais métricas técnicas utilizadas no projeto.

    Parâmetros
    ----------
    y_true : array-like
        Valores reais da variável alvo.

    y_pred : array-like
        Classes previstas pelo modelo.

    y_proba : array-like
        Probabilidades previstas para a classe positiva (Churn = 1).

    Retorna
    -------
    dict
        Dicionário contendo as métricas de avaliação.
    """

    return {
    "Accuracy": accuracy_score(y_true, y_pred),
    "Precision": precision_score(y_true, y_pred, zero_division=0),
    "Recall": recall_score(y_true, y_pred, zero_division=0),
    "F1-Score": f1_score(y_true, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_true, y_proba),
    "Average Precision": average_precision_score(y_true, y_proba)
}

#### Conclusão da Comparação dos Modelos

Foi definido um protocolo único de avaliação para todos os modelos desenvolvidos neste projeto.

O **DummyClassifier** será utilizado como referência mínima de desempenho, enquanto a **Regressão Logística** atuará como baseline estatístico para comparação com os modelos mais avançados.

A seleção do modelo campeão será baseada na análise conjunta das métricas técnicas, da matriz de confusão e dos indicadores econômicos definidos anteriormente, garantindo uma avaliação consistente, reproduzível e alinhada aos objetivos do negócio.

---

In [15]:
# ============================================================
# ETAPA 9 - RESUMO DAS DECISÕES METODOLÓGICAS
# ============================================================
# Consolidando as principais definições que serão utilizadas
# na avaliação dos modelos de previsão de churn.
# ============================================================

resumo_avaliacao = pd.DataFrame({
    "Elemento": [
        "Classe positiva",
        "Divisão dos dados",
        "Estratificação",
        "Random State",
        "Métrica principal",
        "Métricas complementares",
        "Referência mínima",
        "Baseline estatístico",
        "Erro prioritário",
        "Métrica de negócio"
    ],
    "Definição": [
        "Churn = 1 (Cancelou)",
        "80% treino / 20% teste",
        "Preservação da proporção da variável alvo",
        "42",
        "Average Precision (Precision-Recall)",
        "Recall, Precision, F1-Score, ROC-AUC e Accuracy",
        "DummyClassifier",
        "Regressão Logística",
        "Falso Negativo (FN)",
        "Valor Líquido Estimado da Retenção"
    ]
})

display(resumo_avaliacao)

,Elemento,Definição
0,Classe positiva,Churn = 1 (Cancelou)
1,Divisão dos dados,80% treino / 20% teste
2,Estratificação,Preservação da proporção da variável alvo
3,Random State,42
4,Métrica principal,Average Precision (Precision-Recall)
5,Métricas complementares,"Recall, Precision, F1-Score, ROC-AUC e Accuracy"
6,Referência mínima,DummyClassifier
7,Baseline estatístico,Regressão Logística
8,Erro prioritário,Falso Negativo (FN)
9,Métrica de negócio,Valor Líquido Estimado da Retenção


#### Conclusão das Decisões Metodológicas

As definições apresentadas nesta etapa consolidam o protocolo de avaliação que será aplicado de forma consistente a todos os modelos desenvolvidos neste projeto.

A comparação será realizada utilizando o mesmo conjunto de teste, as mesmas métricas técnicas e os mesmos critérios de negócio, garantindo reprodutibilidade e comparabilidade entre os experimentos.

O **DummyClassifier** será utilizado como referência mínima de desempenho, enquanto a **Regressão Logística** atuará como baseline estatístico para comparação com os modelos mais avançados.

---

### ETAPA 10 - Conclusão

Neste notebook foi estabelecido o protocolo metodológico que será utilizado para avaliar todos os modelos de previsão de churn desenvolvidos ao longo deste projeto.

Inicialmente, foi definida a variável alvo e o protocolo de preparação dos dados, garantindo que todos os experimentos utilizem a mesma estrutura de entrada. Em seguida, foi estabelecida uma estratégia de divisão estratificada dos dados, preservando a proporção da classe positiva e assegurando a reprodutibilidade dos resultados por meio do parâmetro `random_state = 42`.

Também foram definidas as métricas técnicas que orientarão a comparação entre os modelos. A **Average Precision (Precision–Recall)** foi adotada como métrica principal por ser mais adequada para problemas de classificação com classes desbalanceadas, enquanto **Recall**, **Precision**, **F1-Score**, **ROC-AUC** e **Accuracy** serão utilizadas como métricas complementares para uma avaliação mais abrangente.

Além dos indicadores estatísticos, foi estabelecida uma abordagem voltada ao impacto no negócio. Os diferentes tipos de erro de classificação foram analisados, com atenção especial aos **Falsos Negativos**, que representam clientes com risco de cancelamento não identificados pelo modelo. Também foi definida uma função parametrizável para estimar o **Valor Líquido da Retenção**, permitindo avaliar o potencial econômico das estratégias de retenção sem assumir valores específicos inexistentes no conjunto de dados.

Por fim, foi formalizado um protocolo de comparação entre modelos. O **DummyClassifier** será utilizado como referência mínima de desempenho, enquanto a **Regressão Logística** atuará como baseline estatístico para comparação com modelos mais avançados, como Random Forest e Redes Neurais.

Com essas definições concluídas, o projeto passa a possuir um protocolo único, consistente e reproduzível para avaliação dos modelos, garantindo que todos os experimentos sejam comparados sob os mesmos critérios técnicos e de negócio.

No próximo notebook será iniciada a etapa de modelagem, com o treinamento, avaliação e análise da **Regressão Logística**, que servirá como primeiro modelo supervisionado de referência para o projeto.

---